# Notebook 1: Data Ingestion, Preprocessing, and Sentiment Extraction

This notebook handles the full data pipeline from raw Reddit and stock price data through to sentiment-scored posts and daily-aggregated datasets. It replicates and extends the methodology from Wang & Luo (2021), which used VADER sentiment to predict GME stock price direction from r/wallstreetbets posts.

**Pipeline overview:**
1. Ingest Reddit WSB posts and GME stock price data
2. Clean and preprocess text following the reference paper's methodology
3. Extract text-based features (word count, stopword count, etc.)
4. Run five sentiment models across all 35,735 posts
5. Aggregate post-level data to daily-level datasets (unweighted and engagement-weighted)

**Data sources:**
- Reddit: [Kaggle WSB Posts](https://www.kaggle.com/datasets/gpreda/reddit-wallstreetsbets-posts) (35,735 posts, Jan 28 – Mar 31, 2021)
- Stock prices: Yahoo Finance via `yfinance` (GME, Jan 4 – Mar 31, 2021)

**Requirements:** This notebook requires a Google Colab environment with a T4 GPU for transformer inference. Sentiment extraction takes approximately 2–3 hours on a T4.

## Environment Setup

In [1]:
!pip install -q emoji vaderSentiment

from google.colab import drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 8.6 MB/s eta 0:00:00
Mounted at /content/drive


In [2]:
import numpy as np
import pandas as pd

import yfinance as yf

import re
import emoji
from tqdm.auto import tqdm

import torch
import nltk
from nltk.corpus import stopwords

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

# Verify GPU availability — required for transformer inference
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected. Transformer inference will be very slow.")

GPU available: True
Device: NVIDIA A100-SXM4-40GB


In [3]:
# File paths
DATA_DIR = "/content/drive/MyDrive/UC Berkeley/Capstone Project/"
SAVE_DIR = "/content/drive/MyDrive/UC Berkeley/Capstone Project/"

## Data Ingestion

The Reddit dataset was collected from r/wallstreetbets via PushShift and published on Kaggle. GME stock data is pulled directly from Yahoo Finance using `yfinance`. The date range (Jan 4 – Mar 31, 2021) covers the GME short squeeze period studied in the reference paper.

In [4]:
# Reddit data: https://www.kaggle.com/datasets/gpreda/reddit-wallstreetsbets-posts
reddit_data = pd.read_csv(DATA_DIR + "reddit_wsb.csv")

# GME stock prices for the study period
gme_data = yf.download("GME", start="2021-01-04", end="2021-04-01")
gme_data.columns = gme_data.columns.get_level_values(0)
gme_data = gme_data.reset_index()

print(f"Reddit posts: {reddit_data.shape}")
print(f"GME trading days: {gme_data.shape}")

/tmp/ipykernel_720/1518905522.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  gme_data = yf.download("GME", start="2021-01-04", end="2021-04-01")
[*********************100%***********************]  1 of 1 completed

Reddit posts: (53187, 8)
GME trading days: (61, 6)


In [5]:
reddit_data.head()

,title,score,id,url,comms_num,created,body,timestamp
0,"It's not about the money, it's about sending a...",55,l6ulcx,https://v.redd.it/6j75regs72e61,6,1.611863e+09,NaN,2021-01-28 21:37:41
1,Math Professor Scott Steiner says the numbers ...,110,l6uibd,https://v.redd.it/ah50lyny62e61,23,1.611862e+09,NaN,2021-01-28 21:32:10
2,Exit the system,0,l6uhhn,https://www.reddit.com/r/wallstreetbets/commen...,47,1.611862e+09,The CEO of NASDAQ pushed to halt trading “to g...,2021-01-28 21:30:35
3,NEW SEC FILING FOR GME! CAN SOMEONE LESS RETAR...,29,l6ugk6,https://sec.report/Document/0001193125-21-019848/,74,1.611862e+09,NaN,2021-01-28 21:28:57
4,"Not to distract from GME, just thought our AMC...",71,l6ufgy,https://i.redd.it/4h2sukb662e61.jpg,156,1.611862e+09,NaN,2021-01-28 21:26:56


In [6]:
gme_data.head()

Price,Date,Close,High,Low,Open,Volume
0,2021-01-04,4.3125,4.7750,4.2875,4.7500,40090000
1,2021-01-05,4.3425,4.5200,4.3075,4.3375,19846000
2,2021-01-06,4.5900,4.7450,4.3325,4.3350,24224800
3,2021-01-07,4.5200,4.8625,4.5050,4.6175,24517200
4,2021-01-08,4.4225,4.5750,4.2700,4.5450,25928000


## Data Cleaning and Feature Engineering

Following the reference paper's methodology, we compute the target variable "Direction" from daily open/close prices, then join Reddit posts to their corresponding trading day. The inner join naturally removes posts made on non-trading days (weekends and holidays). Title and body text are concatenated to form a single text field per post.

In [7]:
df = gme_data[["Date", "Open", "Close"]].copy()

# Target variable: daily price direction
df["Net_Movement"] = df["Close"] - df["Open"]
df["Direction"] = df["Net_Movement"].apply(lambda x: "up" if x > 0 else "down")

# Ensure Date is date-only for clean joining
df["Date"] = pd.to_datetime(df["Date"]).dt.date
reddit_data["Date"] = pd.to_datetime(reddit_data["timestamp"]).dt.date

# Join reddit posts to trading days (inner join removes non-trading days)
df = reddit_data.merge(df, on="Date", how="inner")

# Concatenate title and body as done in the reference paper
df["body"] = df["body"].fillna("")
df["text"] = df["title"] + " " + df["body"]

print(f"Posts matched to trading days: {len(df)}")
print(f"Unique trading days: {df['Date'].nunique()}")
print(f"Direction distribution:\n{df['Direction'].value_counts().to_string()}")

Posts matched to trading days: 35735
Unique trading days: 44
Direction distribution:
Direction
down    28420
up       7315


## Text Preprocessing

The preprocessing pipeline follows the methods outlined in the reference paper (Section 4.1):
1. Replace emojis with descriptive text tags using pipe delimiters (e.g., `|rocket|`)
2. Remove URLs
3. Remove Reddit user mentions (`u/username`)
4. Remove punctuation while preserving decimal numbers and emoji tags
5. Lowercase all text
6. Normalize whitespace

In [8]:
def replace_emojis(text):
    return emoji.demojize(text, delimiters=("|", "|"))

def remove_urls(text):
    return re.sub(r'http\S+|www\.\S+', '', text)

def remove_mentions(text):
    return re.sub(r'u/\S+', '', text)

def remove_punctuation(text):
    # Temporarily protect emoji tags
    protected = re.findall(r'\|[^|]+\|', text)
    for i, tag in enumerate(protected):
        text = text.replace(tag, f'EMOJITAG{i}')
    # Remove punctuation but preserve decimal numbers (e.g., 1.5)
    text = re.sub(r'(?<!\d)\.(?!\d)|[^\w\s.]', '', text)
    text = re.sub(r'(?<!\d)\.(?!\d)', '', text)
    # Restore emoji tags
    for i, tag in enumerate(protected):
        text = text.replace(f'EMOJITAG{i}', tag)
    return text

def normalize_whitespace(text):
    return re.sub(r'\s+', ' ', text).strip()

def preprocess(text):
    if not isinstance(text, str):
        return ""
    text = replace_emojis(text)
    text = remove_urls(text)
    text = remove_mentions(text)
    text = remove_punctuation(text)
    text = text.lower()
    text = normalize_whitespace(text)
    return text

df["text_clean"] = df["text"].apply(preprocess)

## Text Feature Extraction

These features replicate those described in Section 4.3 of the reference paper. They capture surface-level text characteristics that may correlate with sentiment or engagement patterns.

In [9]:
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))

df["word_count"] = df["text_clean"].apply(lambda x: len(x.split()))
df["stopword_count"] = df["text_clean"].apply(lambda x: sum(1 for w in x.split() if w in stop_words))
df["avg_word_length"] = df["text_clean"].apply(lambda x: np.mean([len(w) for w in x.split()]) if x.split() else 0)
df["emoji_count"] = df["text_clean"].apply(lambda x: len(re.findall(r'\|[^|]+\|', x)))

# Drop columns that are no longer needed
df = df.drop(columns=["title", "body", "url", "id", "created", "timestamp", "Open", "Close"])

print(f"DataFrame shape after feature extraction: {df.shape}")
df.info()

DataFrame shape after feature extraction: (35735, 11)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35735 entries, 0 to 35734
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   score            35735 non-null  int64  
 1   comms_num        35735 non-null  int64  
 2   Date             35735 non-null  object 
 3   Net_Movement     35735 non-null  float64
 4   Direction        35735 non-null  object 
 5   text             35735 non-null  object 
 6   text_clean       35735 non-null  object 
 7   word_count       35735 non-null  int64  
 8   stopword_count   35735 non-null  int64  
 9   avg_word_length  35735 non-null  float64
 10  emoji_count      35735 non-null  int64  
dtypes: float64(2), int64(5), object(4)
memory usage: 3.0+ MB


## Sentiment Analysis Models

Five sentiment models are used, each representing a different approach to text analysis:

| Model | Type | Domain | Outputs | Rationale |
|-------|------|--------|---------|-----------|
| **VADER** | Rule-based | General | pos, neu, neg, compound | Paper's original baseline |
| **FinBERT** | Transformer | Financial text | positive, neutral, negative | Financial domain adaptation |
| **Twitter-RoBERTa** | Transformer | Social media | positive, neutral, negative | Social media domain adaptation |
| **Topic-Sentiment RoBERTa** | Transformer | Social media (entity-targeted) | 5-point scale | GME-targeted sentiment |
| **GoEmotions** | Transformer | Reddit | 28 emotion labels | Granular emotion classification |

All transformer models use GPU-accelerated batched inference with `device=0`.

In [10]:
# VADER — rule-based baseline (paper's original method)
vader = SentimentIntensityAnalyzer()

def get_vader_scores(text):
    try:
        scores = vader.polarity_scores(text)
        return scores["pos"], scores["neu"], scores["neg"], scores["compound"]
    except:
        return 0.0, 1.0, 0.0, 0.0

# FinBERT — financial domain sentiment
finbert_pipe = pipeline(
    "sentiment-analysis",
    model="ProsusAI/finbert",
    tokenizer="ProsusAI/finbert",
    top_k=3, truncation=True, max_length=512, device=0
)

# Twitter-RoBERTa — social media sentiment
roberta_pipe = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    tokenizer="cardiffnlp/twitter-roberta-base-sentiment-latest",
    top_k=3, truncation=True, max_length=512, device=0
)

# Topic-Sentiment RoBERTa — entity-targeted sentiment
topic_pipe = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-topic-sentiment-latest",
    tokenizer="cardiffnlp/twitter-roberta-base-topic-sentiment-latest",
    top_k=None, truncation=True, max_length=512, device=0
)

# GoEmotions — 28 emotion labels, trained on Reddit data
goemotions_pipe = pipeline(
    "text-classification",
    model="SamLowe/roberta-base-go_emotions",
    tokenizer="SamLowe/roberta-base-go_emotions",
    top_k=None, truncation=True, max_length=512, device=0
)

EMOTIONS = [
    "admiration", "amusement", "anger", "annoyance", "approval", "caring",
    "confusion", "curiosity", "desire", "disappointment", "disapproval",
    "disgust", "embarrassment", "excitement", "fear", "gratitude", "grief",
    "joy", "love", "nervousness", "optimism", "pride", "realization",
    "relief", "remorse", "sadness", "surprise", "neutral"
]

print("All models loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

All models loaded.


## Sentiment Extraction

All five models are run across the full 35,735 posts. Transformer models use batched inference for efficiency. Checkpoints are saved to Google Drive after each model completes so that progress is preserved if the Colab runtime disconnects.

In [11]:
def run_batched(pipe, texts, batch_size=128):
    """Run a HuggingFace pipeline in batches for GPU efficiency."""
    results = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = [t[:512] for t in texts[i:i+batch_size]]
        results.extend(pipe(batch))
    return results

texts = df["text_clean"].tolist()

# --- VADER (CPU, fast) ---
print("Running VADER...")
vader_results = [get_vader_scores(t) for t in tqdm(texts)]
df["vader_pos"] = [r[0] for r in vader_results]
df["vader_neu"] = [r[1] for r in vader_results]
df["vader_neg"] = [r[2] for r in vader_results]
df["vader_compound"] = [r[3] for r in vader_results]
print("  VADER complete. Saving checkpoint...")
df.to_parquet(SAVE_DIR + "checkpoint_after_vader.parquet", index=False)

# --- FinBERT ---
print("Running FinBERT...")
finbert_results = run_batched(finbert_pipe, texts)
df["finbert_pos"] = [dict((r["label"], r["score"]) for r in res).get("positive", 0) for res in finbert_results]
df["finbert_neu"] = [dict((r["label"], r["score"]) for r in res).get("neutral", 0) for res in finbert_results]
df["finbert_neg"] = [dict((r["label"], r["score"]) for r in res).get("negative", 0) for res in finbert_results]
print("  FinBERT complete. Saving checkpoint...")
df.to_parquet(SAVE_DIR + "checkpoint_after_finbert.parquet", index=False)

# --- Twitter-RoBERTa ---
print("Running Twitter-RoBERTa...")
roberta_results = run_batched(roberta_pipe, texts)
df["roberta_pos"] = [dict((r["label"], r["score"]) for r in res).get("positive", 0) for res in roberta_results]
df["roberta_neu"] = [dict((r["label"], r["score"]) for r in res).get("neutral", 0) for res in roberta_results]
df["roberta_neg"] = [dict((r["label"], r["score"]) for r in res).get("negative", 0) for res in roberta_results]
print("  Twitter-RoBERTa complete. Saving checkpoint...")
df.to_parquet(SAVE_DIR + "checkpoint_after_roberta.parquet", index=False)

# --- Topic-Sentiment RoBERTa ---
print("Running Topic-Sentiment RoBERTa...")
topic_texts = [f"{t[:450]} </s> GME" for t in texts]
topic_results = run_batched(topic_pipe, topic_texts)
df["topic_strong_pos"] = [dict((r["label"], r["score"]) for r in res).get("strongly positive", 0) for res in topic_results]
df["topic_pos"] = [dict((r["label"], r["score"]) for r in res).get("positive", 0) for res in topic_results]
df["topic_neu"] = [dict((r["label"], r["score"]) for r in res).get("negative or neutral", 0) for res in topic_results]
df["topic_neg"] = [dict((r["label"], r["score"]) for r in res).get("negative", 0) for res in topic_results]
df["topic_strong_neg"] = [dict((r["label"], r["score"]) for r in res).get("strongly negative", 0) for res in topic_results]
print("  Topic-Sentiment complete. Saving checkpoint...")
df.to_parquet(SAVE_DIR + "checkpoint_after_topic.parquet", index=False)

# --- GoEmotions ---
print("Running GoEmotions...")
goemotions_results = run_batched(goemotions_pipe, texts)
for idx, emotion in enumerate(EMOTIONS):
    df[f"emo_{emotion}"] = [dict((r["label"], r["score"]) for r in res).get(emotion, 0) for res in goemotions_results]

print("\n" + "=" * 60)
print("ALL MODELS COMPLETE")
print(f"Final dataframe shape: {df.shape}")
print("=" * 60)

# Save Dataset 1: Post-level with all sentiments
df.to_parquet(SAVE_DIR + "df_with_all_sentiments.parquet", index=False)

Running VADER...


  0%|          | 0/35735 [00:00<?, ?it/s]

  VADER complete. Saving checkpoint...
Running FinBERT...


  0%|          | 0/280 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


...

  Twitter-RoBERTa complete. Saving checkpoint...
Running Topic-Sentiment RoBERTa...


  0%|          | 0/280 [00:00<?, ?it/s]

  Topic-Sentiment complete. Saving checkpoint...
Running GoEmotions...


  0%|          | 0/280 [00:00<?, ?it/s]


ALL MODELS COMPLETE
Final dataframe shape: (35735, 54)


## Daily Aggregation

The reference paper trained classifiers at the post level, treating each of the 35,735 posts as an independent observation. However, all posts on the same trading day share the same Direction label, which creates a data leakage risk when using random train/test splits. To address this, we aggregate features to the daily level, producing one row per trading day (n=44).

Two aggregation strategies are tested:
- **Unweighted (Dataset 2):** Simple mean and standard deviation of all features per day
- **Engagement-weighted (Dataset 3):** Sentiment features weighted by upvote score, under the hypothesis that highly upvoted posts better represent community sentiment. Metadata and text features use unweighted aggregation to avoid circular weighting.

In [12]:
# Define feature groups
metadata_features = ["score", "comms_num"]
text_features = ["word_count", "stopword_count", "avg_word_length", "emoji_count"]

vader_features = ["vader_pos", "vader_neu", "vader_neg", "vader_compound"]
finbert_features = ["finbert_pos", "finbert_neu", "finbert_neg"]
roberta_features = ["roberta_pos", "roberta_neu", "roberta_neg"]
topic_features = ["topic_strong_pos", "topic_pos", "topic_neu", "topic_neg", "topic_strong_neg"]
goemotions_features = [
    "emo_admiration", "emo_amusement", "emo_anger", "emo_annoyance", "emo_approval",
    "emo_caring", "emo_confusion", "emo_curiosity", "emo_desire", "emo_disappointment",
    "emo_disapproval", "emo_disgust", "emo_embarrassment", "emo_excitement", "emo_fear",
    "emo_gratitude", "emo_grief", "emo_joy", "emo_love", "emo_nervousness",
    "emo_optimism", "emo_pride", "emo_realization", "emo_relief", "emo_remorse",
    "emo_sadness", "emo_surprise", "emo_neutral"
]

all_numeric_features = (
    metadata_features + text_features +
    vader_features + finbert_features + roberta_features +
    topic_features + goemotions_features
)

weighted_features = (
    vader_features + finbert_features + roberta_features +
    topic_features + goemotions_features
)

# --- Dataset 2: Daily Aggregated (Unweighted) ---
print("Building Dataset 2: Daily Aggregated (Unweighted)...")

mean_agg = df.groupby("Date")[all_numeric_features].mean()
mean_agg.columns = [f"{c}_mean" for c in mean_agg.columns]

std_agg = df.groupby("Date")[all_numeric_features].std()
std_agg.columns = [f"{c}_std" for c in std_agg.columns]

post_counts = df.groupby("Date").size().rename("post_count")

targets = df.groupby("Date").agg(
    Direction=("Direction", "first"),
    Net_Movement=("Net_Movement", "first")
)

df_daily = pd.concat([targets, post_counts, mean_agg, std_agg], axis=1).reset_index()
df_daily = df_daily.sort_values("Date").reset_index(drop=True)

print(f"  Shape: {df_daily.shape}")
print(f"  Date range: {df_daily['Date'].min()} to {df_daily['Date'].max()}")
print(f"  Direction distribution:\n{df_daily['Direction'].value_counts().to_string()}")

# --- Dataset 3: Daily Aggregated (Weighted) ---
print("\nBuilding Dataset 3: Daily Aggregated (Weighted)...")

def weighted_mean(group, features, weight_col="score"):
    weights = group[weight_col].clip(lower=1)
    result = {}
    for f in features:
        result[f"{f}_wmean"] = np.average(group[f], weights=weights)
    return pd.Series(result)

def weighted_std(group, features, weight_col="score"):
    weights = group[weight_col].clip(lower=1)
    result = {}
    for f in features:
        avg = np.average(group[f], weights=weights)
        variance = np.average((group[f] - avg) ** 2, weights=weights)
        result[f"{f}_wstd"] = np.sqrt(variance)
    return pd.Series(result)

weighted_mean_agg = df.groupby("Date").apply(lambda g: weighted_mean(g, weighted_features))
weighted_std_agg = df.groupby("Date").apply(lambda g: weighted_std(g, weighted_features))

non_weighted_mean = df.groupby("Date")[metadata_features + text_features].mean()
non_weighted_mean.columns = [f"{c}_mean" for c in non_weighted_mean.columns]

non_weighted_std = df.groupby("Date")[metadata_features + text_features].std()
non_weighted_std.columns = [f"{c}_std" for c in non_weighted_std.columns]

df_daily_weighted = pd.concat([
    targets, post_counts,
    non_weighted_mean, non_weighted_std,
    weighted_mean_agg, weighted_std_agg
], axis=1).reset_index()

df_daily_weighted = df_daily_weighted.sort_values("Date").reset_index(drop=True)

print(f"  Shape: {df_daily_weighted.shape}")

# Save
df_daily.to_parquet(SAVE_DIR + "dataset2_daily_unweighted.parquet", index=False)
df_daily_weighted.to_parquet(SAVE_DIR + "dataset3_daily_weighted.parquet", index=False)

print("\n" + "=" * 60)
print("ALL DATASETS SAVED")
print(f"  Dataset 1 (post-level):       {df.shape}")
print(f"  Dataset 2 (daily unweighted): {df_daily.shape}")
print(f"  Dataset 3 (daily weighted):   {df_daily_weighted.shape}")
print("=" * 60)

Building Dataset 2: Daily Aggregated (Unweighted)...
  Shape: (44, 102)
  Date range: 2021-01-28 to 2021-03-31
  Direction distribution:
Direction
down    26
up      18

Building Dataset 3: Daily Aggregated (Weighted)...


/tmp/ipykernel_720/3737650147.py:71: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  weighted_mean_agg = df.groupby("Date").apply(lambda g: weighted_mean(g, weighted_features))
/tmp/ipykernel_720/3737650147.py:72: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  weighted_std_agg = df.groupby("Date").apply(lambda g: weighted_std(g, weighted_features))


  Shape: (44, 102)

ALL DATASETS SAVED
  Dataset 1 (post-level):       (35735, 54)
  Dataset 2 (daily unweighted): (44, 102)
  Dataset 3 (daily weighted):   (44, 102)
